# 02_Quality_Control_and_Preprocessing

## Objectives
1. **Quality Assessment**: Run `FastQC` on all raw FASTQ files in parallel or loop to evaluate per-base sequence quality, adapter content, and GC bias.
2. **Aggregation**: Use `MultiQC` to combine individual FastQC reports into a single, interactive HTML dashboard.
3. **Primer Clipping**: Utilize `cutadapt` to remove specific 16S rRNA primers prior to downstream inference.

In [ ]:
import os
import subprocess

# 1. Ensure qc_reports directory exists
output_dir = "qc_reports"
os.makedirs(output_dir, exist_ok=True)

# 2. Check and install MultiQC if missing
print("Checking MultiQC installation...")
multiqc_check = subprocess.run("which multiqc", shell=True, capture_output=True, text=True)
if multiqc_check.returncode != 0:
    print("Multiqc not found. Installing via pip...")
    subprocess.run("pip install multiqc", shell=True, check=True)
    print("MultiQC installed successfully!")
else:
    print("MultiQC is already installed.")

# 3. Retrieve all FASTQ files
input_dir = "raw_data"
fastq_files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith(".fastq")]
print(f"Total Fastq files to process: {len(fastq_files)}")

# 4. Run FastQC on ALL samples
print("Starting FastQC processing for all samples (this may take a while)...")
for i, fastq in enumerate(fastq_files, 1):
    print(f"[{i}/{len(fastq_files)}] Running FastQC on: {os.path.basename(fastq)}")
    cmd = f"fastqc {fastq} -o {output_dir}/"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"Warning: FastQC failed for {os.path.basename(fastq)}")

print("All FastQC reports generated successfully!")

# 5. Run MultiQC to aggregate all reports into a single dashboard
print("Running MultiQC to aggregate reports...")
multiqc_cmd = f"multiqc {output_dir}/ -o {output_dir}/"
subprocess.run(multiqc_cmd, shell=True, check=True)

print("MultiQC aggregation completed! Check the 'qc_reports' directory for the final HTML summary.")

## 3. Primer Removal via Cutadapt
In 16S rRNA gene amplicon sequencing, PCR primers are attached to the ends of the target sequences. Retaining these primers can interfere with accurate taxonomic classification and downstream ASV inference (e.g., in DADA2). 

### Objectives:
* **Targeted Clipping**: Remove the specific 16S rRNA PCR primers from the reads.
* **Length Filtering**: Filter out reads that become excessively short after trimming (using `--minimum-length 50`).
* **Output Preparation**: Save the cleaned, primer-free reads into a dedicated `trimmed_data/` directory for downstream processing.

In [ ]:
import os
import subprocess

# 1. Create directory for trimmed reads
trimmed_dir = "trimmed_data"
os.makedirs(trimmed_dir, exist_ok=True)

# 2. Define target 16S rRNA forward primer sequence
forward_primer = "CCTACGGGNGGCWGCAG"
input_dir = "raw_data"
fastq_files = [f for f in os.listdir(input_dir) if f.endswith(".fastq")]

# 3. Specify the absolute path to cutadapt within the isolated conda environment
cutadapt_path = "/home/azureuser/miniconda3/envs/cutadapt_env/bin/cutadapt"

print(f"Starting full primer trimming for {len(fastq_files)} samples...")

# 4. Loop through all fastq files and apply cutadapt trimming
for i, filename in enumerate(fastq_files, 1):
    input_path = os.path.join(input_dir, filename)
    output_path = os.path.join(trimmed_dir, filename)
    
    # Run cutadapt: remove 5' primer and filter reads shorter than 50 bp
    cmd = f"{cutadapt_path} -g {forward_primer} -o {output_path} {input_path} --minimum-length 50"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    # Print progress milestones
    if i % 100 == 0 or i == len(fastq_files):
        print(f"Progress: Processed {i}/{len(fastq_files)} samples...")

print("All samples trimmed successfully! Cleaned files ready in 'trimmed_data/'.")

In [ ]:
# ==========================================
# Generate Cutadapt Log Files for MultiQC
# ==========================================
import os
import subprocess

# Define directories and parameters
input_dir = "raw_data"
trimmed_dir = "trimmed_data"
logs_dir = "cutadapt_logs"
os.makedirs(logs_dir, exist_ok=True)
os.makedirs(trimmed_dir, exist_ok=True)

forward_primer = "CCTACGGGNGGCWGCAG"
cutadapt_path = "/home/azureuser/miniconda3/envs/cutadapt_env/bin/cutadapt"
fastq_files = [f for f in os.listdir(input_dir) if f.endswith(".fastq")]

print(f"Generating log files for {len(fastq_files)} samples...")

# Loop through all files to regenerate logs without issues
for i, filename in enumerate(fastq_files, 1):
    input_path = os.path.join(input_dir, filename)
    output_path = os.path.join(trimmed_dir, filename)
    log_path = os.path.join(logs_dir, filename.replace(".fastq", ".log"))
    
    # Run cutadapt and redirect output stream to the individual log file
    cmd = f"{cutadapt_path} -g {forward_primer} -o {output_path} {input_path} --minimum-length 50 > {log_path} 2>&1"
    subprocess.run(cmd, shell=True)
    
    # Print progress milestones
    if i % 100 == 0 or i == len(fastq_files):
        print(f"Progress: Generated {i}/{len(fastq_files)} logs...")

print("All log files generated successfully!")